# 03 · Cleaning

Phase 4: fix the problems found in `02_eda.ipynb`. The steps live in `src/data.py` (`clean_slice`) so the baseline and evaluation notebooks use exactly the same cleaned data.

In [1]:
import sys
sys.path.append("../src")

import re
import pandas as pd
from data import load_case_law, load_slice, clean_slice

In [2]:
case_law = load_case_law()
raw = load_slice(case_law)
clean, log = clean_slice(raw, case_law)

## 1. Before/after counts

In [3]:
log

,step,rows before,rows after
0,drop chunks from page-header-only judgments,400,381
1,drop exact duplicate text,381,381
2,strip leading/trailing whitespace (92 chunks c...,381,381
3,replace non-breaking spaces in title (381 titl...,381,381


In [4]:
print("raw slice:    ", len(raw), "chunks from", raw["url"].nunique(), "judgments")
print("cleaned slice:", len(clean), "chunks from", clean["url"].nunique(), "judgments")

raw slice:     400 chunks from 272 judgments
cleaned slice: 381 chunks from 253 judgments


## 2. What each step does, and why

**1. Drop chunks from page-header-only judgments (400 → 381).** These 19 chunks are the Kenya Law web-page header (court, judges, "Download DOCX", "Loading PDF…"), not judgment text. They are packed with neatly formatted dates and case numbers, so keeping them would make the extractor look better than it is on real judgments. The whole judgment is dropped, not only the chunk containing "Loading PDF", because the chunk before it is the same header.

In [5]:
dropped = raw[~raw["chunk_id"].isin(clean["chunk_id"])]
print(dropped["date"].eq("").all(), "- every dropped chunk has an empty date field")
dropped["text"].str[:110].head(4).tolist()

True - every dropped chunk has an empty date field


['E218 of 2024 Judges SG Kairu, KI Laibuta, GW Ngenye-Macharia Judgment date 25 March 2026 Language English Type',
 'M’Muguongo & another v Mugambi (Environment and Land Appeal E080 of 2025) [2026] KEELC 1870 (KLR) (25 March 20',
 'zoia [2026] KEELC 1747 (KLR) Copy Media Neutral Citation [2026] KEELC 1747 (KLR) Copy Court Environment and La',
 'se number Election Petition E002 of 2025 Judges RM Mwongo Judgment date 27 March 2026 Language English Type Ju']

**2. Drop exact duplicate text (381 → 381).** No duplicates made it into this sample. The step stays in so the cleaning would still be correct on a different sample: the full case-law set has 20 repeated texts.

**3. Strip leading/trailing whitespace.** No rows are removed. This matters because extracted entities are compared by character position and string later, and stray spaces at chunk edges would cause false mismatches.

In [6]:
before = raw.set_index("chunk_id").loc[clean["chunk_id"], "text"]
changed = before[before != before.str.strip()]
[repr(t[:40]) for t in changed.head(3)]

["'N along the aforesaid road, when the def'",
 "' of administration ad litem. That he bel'",
 "'a huge leap of faith. It explains why it'"]

**4. Replace non-breaking spaces (`\xa0`) in titles.** Every title uses `\xa0` between words (e.g. `Case E827\xa0of\xa02025`). They look like spaces but aren't, so a pattern containing a plain space silently fails to match. The chunk text doesn't contain any, so only titles change.

In [7]:
print(repr(raw["title"].iloc[0]))
print(repr(clean["title"].iloc[0]))

'Sikale v Kichachu & 2 others (Environment & Land Case E013\xa0of\xa02021) [2024]\xa0KEMC\xa050\xa0(KLR) (9\xa0October\xa02024) (Judgment)'
'Sikale v Kichachu & 2 others (Environment & Land Case E013 of 2021) [2024] KEMC 50 (KLR) (9 October 2024) (Judgment)'


## 3. What was deliberately *not* cleaned

These were found in the EDA and left alone on purpose. The baseline has to cope with real judgment text, and cleaning them away would hide the weaknesses Phase 6 is supposed to measure.

In [8]:
pd.DataFrame({
    "left as is": [
        "chunks starting with a lowercase letter (cut mid-sentence)",
        "curly quotes and ellipsis characters",
        "spaced ordinals, e.g. '1 st Defendant', '13 th August 2016'",
        "'/=' after amounts, e.g. 'Ksh. 80,000/='",
        "court codes that look like money, e.g. 'KESC'",
    ],
    "chunks in cleaned slice": [
        clean["text"].str.match(r"[a-z]").sum(),
        clean["text"].str.contains("[\u2018\u2019\u201c\u201d\u2026]").sum(),
        clean["text"].str.contains(r"\d+ (?:TH|ST|ND|RD)\b", case=False).sum(),
        clean["text"].str.contains("/=").sum(),
        clean["text"].str.contains(r"\bKESC\b").sum(),
    ],
    "why kept": [
        "rejoining would change the unit of analysis; boundary splits are a real limitation to report",
        "don't affect any of the target patterns",
        "mostly party labels, but some are dates that a 'DD Month YYYY' pattern misses",
        "a real Kenyan money convention the extractor should handle",
        "a realistic false-positive trap for a naive money pattern",
    ],
})

,left as is,chunks in cleaned slice,why kept
0,chunks starting with a lowercase letter (cut m...,330,rejoining would change the unit of analysis; b...
1,curly quotes and ellipsis characters,218,don't affect any of the target patterns
2,"spaced ordinals, e.g. '1 st Defendant', '13 th...",114,"mostly party labels, but some are dates that a..."
3,"'/=' after amounts, e.g. 'Ksh. 80,000/='",29,a real Kenyan money convention the extractor s...
4,"court codes that look like money, e.g. 'KESC'",2,a realistic false-positive trap for a naive mo...


The lowercase-start count is higher than the 291 in the EDA because stripping leading spaces exposed more chunks that begin mid-sentence (the EDA check needed the first character to be a letter).

The spaced ordinals matter more than they look. Most are party labels such as *1 st Defendant*, but dates are written the same way (*13 th August 2016*, *23 RD DAY OF MARCH*). The written-date pattern above expects `13th August`, so these dates will be missed. That's a failure case to check for in Phase 6.

## 4. Pattern frequency after cleaning

The same rough counting patterns as in the EDA, run on the cleaned slice, to show what the cleaning changed.

In [9]:
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"

patterns = {
    "date: 13/7/2022":            r"\b\d{1,2}[/.-]\d{1,2}[/.-]\d{2,4}\b",
    "date: 25 March 2026":        rf"\b\d{{1,2}}(?:st|nd|rd|th)?\s+(?:{MONTHS}),?\s+\d{{4}}\b",
    "date: March 25, 2026":       rf"\b(?:{MONTHS})\s+\d{{1,2}},\s+\d{{4}}\b",
    "money: Kshs. 80,000":        r"\b(?:KES|Kshs?|KSh)\.?\s?\d[\d,]*",
    "case ref: No. 27 of 2010":   r"\b(?:No\.?\s*)?E?\d+\s+of\s+\d{4}\b",
    "case ref: [2026] KEHC 3922": r"\[\d{4}\]\s*[A-Za-z]+\s*\d+",
}

def chunks_with(frame, pattern):
    return frame["text"].str.contains(pattern, flags=re.IGNORECASE).sum()

pd.DataFrame({
    "raw (400)": {name: chunks_with(raw, p) for name, p in patterns.items()},
    "cleaned (381)": {name: chunks_with(clean, p) for name, p in patterns.items()},
})

,raw (400),cleaned (381)
date: 13/7/2022,40,40
date: 25 March 2026,45,27
"date: March 25, 2026",13,13
"money: Kshs. 80,000",52,52
case ref: No. 27 of 2010,65,48
case ref: [2026] KEHC 3922,60,47


The written-date and case-reference counts fall the most. That fits what was removed: page headers contain a judgment date and a case number by design. What remains is the text the baseline will be evaluated on.

## Summary

- **Cleaned slice: 381 chunks** from the original 400. All 19 removed chunks came from judgments where only the page header was scraped.
- **No duplicates** in the sample, but the check stays in place.
- **Formatting fixes** (whitespace, `\xa0`) change no row counts.
- **Real-world messiness is kept on purpose** so the baseline in Phase 5 is tested on the text a lawyer would actually give it.